In [1]:
import pandas as pd

df = pd.read_csv("../delhi-aqi-prediction-using-ml/data/delhi_final_clean.csv")
print(df.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_11472\3171985134.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


              datetime    pm25    pm10    no2   so2     co     o3    aqi  \
0  2015-01-01 01:00:00  454.58  935.18  41.78  3.41   9.29  54.94  456.0   
1  2015-01-01 02:00:00  440.44  935.18  43.46  3.88  13.28  50.53  456.0   
2  2015-01-01 03:00:00  409.09  935.18  41.19  2.83  29.67  19.33  456.0   
3  2015-01-01 04:00:00  436.12  935.18  39.55  4.33  21.76  20.08  456.0   
4  2015-01-01 05:00:00  415.88  976.99  37.41  6.17  26.19  16.00  456.0   

   hour  day  month  dayofweek  
0   1.0  1.0    1.0        3.0  
1   2.0  1.0    1.0        3.0  
2   3.0  1.0    1.0        3.0  
3   4.0  1.0    1.0        3.0  
4   5.0  1.0    1.0        3.0  


In [2]:
df["datetime"] = pd.to_datetime(df["datetime"])

df = df.sort_values("datetime")

df.set_index("datetime", inplace=True)

In [3]:
print(type(df.index))

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [4]:
ts = df["aqi"]

In [5]:
#fixing missing timestamps
ts = ts.resample("H").mean()
ts = ts.ffill()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_11472\3186661142.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ts = ts.resample("H").mean()


In [6]:
train = ts[:'2021-12-31']
test = ts['2022-01-01':]

In [7]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    train,
    order=(2,1,2),
    seasonal_order=(1,1,1,24)
)

model_fit = model.fit()

MemoryError: Unable to allocate 1.24 GiB for an array with shape (52, 52, 61367) and data type float64

In [ ]:
forecast = model_fit.forecast(steps=len(test))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))

print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(test.values, label="Actual")
plt.plot(forecast.values, label="Predicted")
plt.legend()
plt.show()

In [ ]:
# prepare RF dataset with datetime filter
df_rf = df.copy()

train_rf = df_rf[df_rf.index < "2022-01-01"]
test_rf = df_rf[df_rf.index >= "2022-01-01"]

X_train = train_rf[["pm25","pm10","no2","so2","co","o3"]]
y_train = train_rf["aqi"]

X_test = test_rf[["pm25","pm10","no2","so2","co","o3"]]
y_test = test_rf["aqi"]

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# SARIMA
sarima_mae = mean_absolute_error(test, sarima_pred)
sarima_rmse = np.sqrt(mean_squared_error(test, sarima_pred))
sarima_r2 = r2_score(test, sarima_pred)

# RF
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["SARIMA", "Random Forest"],
    "R2 Score": [sarima_r2, rf_r2],
    "MAE": [sarima_mae, rf_mae],
    "RMSE": [sarima_rmse, rf_rmse]
})

comparison = comparison.round(3)
print(comparison)